In [13]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

### Data import

In [10]:
df = pd.read_excel('../data/raw/Intraday_vol.xlsx')
df.head()

,Date,Open,High,Low,Close
0,2026-03-24 16:00:00,6556.48,6556.48,6556.33,6556.37
1,2026-03-24 15:30:00,6569.94,6573.06,6551.46,6557.19
2,2026-03-24 15:00:00,6574.44,6582.09,6569.52,6570.00
3,2026-03-24 14:30:00,6570.98,6590.67,6569.27,6574.42
4,2026-03-24 14:00:00,6567.68,6574.38,6558.01,6570.92


In [11]:
df_close = df[['Date', 'Close']].copy()
df_close['Date'] = pd.to_datetime(df_close['Date'])
df_close = df_close.sort_values('Date').reset_index(drop=True)
df_close['trade_date'] = df_close['Date'].dt.date


,Date,Close,trade_date
0,2025-07-07 09:30:00,6258.81,2025-07-07
1,2025-07-07 10:00:00,6252.43,2025-07-07
2,2025-07-07 10:30:00,6243.69,2025-07-07
3,2025-07-07 11:00:00,6242.60,2025-07-07
4,2025-07-07 11:30:00,6239.54,2025-07-07
5,2025-07-07 12:00:00,6227.93,2025-07-07
6,2025-07-07 12:30:00,6220.22,2025-07-07
7,2025-07-07 13:00:00,6223.57,2025-07-07
8,2025-07-07 13:30:00,6222.48,2025-07-07
9,2025-07-07 14:00:00,6201.65,2025-07-07


In [14]:
# Calculate log returns by grouping by 'trade_date'
df_close['log_return'] = (
    df_close.groupby('trade_date')['Close']
    .transform(lambda x: np.log(x / x.shift(1)))
)
df_close['sq_log_return'] = df_close['log_return'] ** 2
df_close.head()

,Date,Close,trade_date,log_return,sq_log_return
0,2025-07-07 09:30:00,6258.81,2025-07-07,NaN,NaN
1,2025-07-07 10:00:00,6252.43,2025-07-07,-0.001020,1.040161e-06
2,2025-07-07 10:30:00,6243.69,2025-07-07,-0.001399,1.956738e-06
3,2025-07-07 11:00:00,6242.60,2025-07-07,-0.000175,3.048219e-08
4,2025-07-07 11:30:00,6239.54,2025-07-07,-0.000490,2.403946e-07


In [15]:
daily_intraday_vol = (
    df_close.groupby('trade_date')['sq_log_return']
    .mean()
    .pipe(np.sqrt)
    .reset_index(name='intraday_avg_vol')
)
print(daily_intraday_vol.head())

   trade_date  intraday_avg_vol
0  2025-07-07          0.001718
1  2025-07-08          0.000906
2  2025-07-09          0.001095
3  2025-07-10          0.000713
4  2025-07-11          0.000620


In [16]:
daily_intraday_vol.to_csv("../data/processed/SPX_Intraday_Vol.csv", index=False)